# Laboratório — Estatística descritiva, robustez e outliers

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/02-statistics/notebooks/11-descritiva-robustez-outliers-laboratorio.ipynb)

**Objetivo:** descrever a massa corporal do Palmer Penguins, comparar medidas clássicas e robustas e investigar uma corrupção sintética sem alterar os dados brutos.

> O valor 63.000 g criado adiante é artificial e existe somente em uma cópia. Nenhuma observação real é rotulada como erro.

## Dependências e reprodutibilidade

- Python 3.10 ou superior
- NumPy
- pandas
- SciPy
- Matplotlib

O laboratório não depende de APIs privadas. No repositório, lê o CSV versionado; no Colab, usa como fallback o arquivo bruto da branch `main`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260907
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)

## 1. Carregamento e auditoria inicial

A lista de caminhos permite executar o notebook tanto da raiz do repositório quanto da pasta de notebooks. O fallback remoto é útil no Colab e aponta para o mesmo CSV versionado.

In [ ]:
candidatos = [
    Path("02-statistics/datasets/11-palmer-penguins.csv"),
    Path("../datasets/11-palmer-penguins.csv"),
]
url_fallback = (
    "https://raw.githubusercontent.com/joaopaulomirandamatias/"
    "ai-lab/main/02-statistics/datasets/11-palmer-penguins.csv"
)

caminho = next((p for p in candidatos if p.exists()), None)
origem = str(caminho) if caminho else url_fallback
df_bruto = pd.read_csv(origem, na_values="NA")

print("Origem:", origem)
print("Dimensão:", df_bruto.shape)
df_bruto.head()

In [ ]:
colunas_esperadas = [
    "species", "island", "bill_length_mm", "bill_depth_mm",
    "flipper_length_mm", "body_mass_g", "sex", "year",
]
assert df_bruto.shape == (344, 8)
assert df_bruto.columns.tolist() == colunas_esperadas
assert (df_bruto["body_mass_g"].dropna() > 0).all()
assert set(df_bruto["species"]) == {"Adelie", "Chinstrap", "Gentoo"}
assert df_bruto.duplicated().sum() == 0

auditoria = pd.DataFrame({
    "tipo": df_bruto.dtypes.astype(str),
    "ausentes": df_bruto.isna().sum(),
    "valores_unicos": df_bruto.nunique(dropna=True),
})
auditoria

Há 344 unidades observacionais, mas duas massas ausentes. Toda estatística de massa deve informar (n=342). As ausências não são preenchidas automaticamente.

In [ ]:
x = df_bruto["body_mass_g"].dropna().to_numpy(dtype=float)
assert x.size == 342
print(f"Linhas totais: {len(df_bruto)}")
print(f"Massas válidas: {x.size}")
print(f"Massas ausentes: {df_bruto['body_mass_g'].isna().sum()}")

## 2. Resumos clássicos e robustos

A função abaixo declara `ddof=1`, o método dos quantis e as duas convenções de MAD. Isso torna o cálculo auditável.

In [ ]:
def resumo_distribuicao(valores):
    a = np.asarray(valores, dtype=float)
    q1, mediana, q3 = np.quantile(a, [0.25, 0.50, 0.75], method="linear")
    return pd.Series({
        "n": a.size,
        "mínimo": a.min(),
        "Q1": q1,
        "média": a.mean(),
        "mediana": mediana,
        "Q3": q3,
        "máximo": a.max(),
        "desvio-padrão (ddof=1)": a.std(ddof=1),
        "IQR": q3 - q1,
        "MAD bruto": stats.median_abs_deviation(a, scale=1),
        "MAD escalado normal": stats.median_abs_deviation(a, scale="normal"),
        "média aparada 10%": stats.trim_mean(a, 0.10),
    })

resumo_original = resumo_distribuicao(x)
resumo_original.to_frame("massa corporal (g)").round(4)

In [ ]:
assert np.isclose(resumo_original["média"], 4201.754385964912)
assert resumo_original["mediana"] == 4050
assert np.isclose(resumo_original["desvio-padrão (ddof=1)"], 801.9545356980955)
assert resumo_original["IQR"] == 1200
assert resumo_original["MAD bruto"] == 600
assert np.isclose(resumo_original["MAD escalado normal"], 889.5613311033611)
print("Asserções dos resumos: OK")

## 3. Histograma, ECDF e boxplot

Nenhum gráfico conta a história inteira. A regra de Freedman–Diaconis define a largura das classes do histograma; a ECDF não exige classes; o boxplot resume quartis e cercas.

In [ ]:
iqr = stats.iqr(x)
largura_fd = 2 * iqr * x.size ** (-1 / 3)
n_classes = int(np.ceil((x.max() - x.min()) / largura_fd))

x_ordenado = np.sort(x)
ecdf = np.arange(1, x.size + 1) / x.size

print(f"Largura de Freedman–Diaconis: {largura_fd:.3f} g")
print(f"Número de classes: {n_classes}")

In [ ]:
azul, laranja, verde = "#0072B2", "#E69F00", "#009E73"
fig, eixos = plt.subplots(1, 3, figsize=(16, 4.5))

eixos[0].hist(x, bins=n_classes, color=azul, edgecolor="white")
eixos[0].set(title="Histograma da massa corporal", xlabel="Massa (g)", ylabel="Contagem")

eixos[1].step(x_ordenado, ecdf, where="post", color=laranja, linewidth=2)
eixos[1].set(title="ECDF da massa corporal", xlabel="Massa (g)", ylabel="Proporção acumulada")
eixos[1].grid(alpha=0.25)

eixos[2].boxplot(x, vert=True, patch_artist=True,
                 boxprops={"facecolor": verde, "alpha": 0.65})
eixos[2].set(title="Boxplot global", ylabel="Massa (g)", xticks=[1], xticklabels=["Todos"])

fig.suptitle("Três visões complementares — Palmer Penguins", fontweight="bold")
fig.tight_layout()
plt.show()

## 4. O contexto de grupo

A distribuição global combina espécies com massas muito diferentes. Resumos por grupo ajudam a distinguir composição de anomalia.

In [ ]:
por_especie = (
    df_bruto.groupby("species", observed=True)["body_mass_g"]
    .agg(n="count", média="mean", mediana="median", desvio_padrão="std", mínimo="min", máximo="max")
    .round(2)
)
por_especie

In [ ]:
ordem = ["Adelie", "Chinstrap", "Gentoo"]
dados_grupos = [
    df_bruto.loc[df_bruto["species"] == especie, "body_mass_g"].dropna()
    for especie in ordem
]
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(dados_grupos, labels=ordem, patch_artist=True)
for caixa, cor in zip(bp["boxes"], [azul, laranja, verde]):
    caixa.set_facecolor(cor)
    caixa.set_alpha(0.65)
ax.set(title="Massa corporal por espécie", xlabel="Espécie", ylabel="Massa (g)")
ax.grid(axis="y", alpha=0.25)
plt.show()

assert por_especie.loc["Adelie", "n"] == 151
assert por_especie.loc["Chinstrap", "n"] == 68
assert por_especie.loc["Gentoo", "n"] == 123

## 5. Corrupção sintética controlada

Criamos uma cópia e trocamos somente o maior valor, 6.300 g, por 63.000 g. O DataFrame `df_bruto` permanece intacto.

In [ ]:
df_sintetico = df_bruto.copy(deep=True)
indice_alvo = df_sintetico["body_mass_g"].idxmax()
valor_original = df_sintetico.loc[indice_alvo, "body_mass_g"]
df_sintetico.loc[indice_alvo, "body_mass_g"] = 63_000.0
df_sintetico["flag_corrupcao_sintetica"] = False
df_sintetico.loc[indice_alvo, "flag_corrupcao_sintetica"] = True

assert valor_original == 6300
assert df_bruto.loc[indice_alvo, "body_mass_g"] == 6300
assert df_sintetico.loc[indice_alvo, "body_mass_g"] == 63000

df_sintetico.loc[[indice_alvo], ["species", "island", "body_mass_g", "flag_corrupcao_sintetica"]]

In [ ]:
x_corrompido = df_sintetico["body_mass_g"].dropna().to_numpy()
comparacao = pd.concat(
    [resumo_distribuicao(x), resumo_distribuicao(x_corrompido)],
    axis=1,
)
comparacao.columns = ["original", "cópia corrompida"]
comparacao.round(4)

In [ ]:
assert np.isclose(comparacao.loc["média", "cópia corrompida"], 4367.543859649123)
assert comparacao.loc["mediana", "cópia corrompida"] == 4050
assert comparacao.loc["IQR", "cópia corrompida"] == 1200
assert comparacao.loc["MAD bruto", "cópia corrompida"] == 600
assert np.isclose(comparacao.loc["desvio-padrão (ddof=1)", "cópia corrompida"], 3277.3722069845344)
print("A média subiu {:.2f} g; a mediana não mudou.".format(
    comparacao.loc["média", "cópia corrompida"] - comparacao.loc["média", "original"]
))

## 6. Sinalização: cercas de Tukey e z modificado

As funções retornam flags para investigação, nunca ordens de exclusão. O z modificado usa o **MAD bruto**. Se MAD for zero, retorna valores ausentes em vez de dividir por zero.

In [ ]:
def flags_tukey(valores, fator=1.5):
    a = np.asarray(valores, dtype=float)
    q1, q3 = np.quantile(a, [0.25, 0.75], method="linear")
    amplitude = q3 - q1
    inferior, superior = q1 - fator * amplitude, q3 + fator * amplitude
    return (a < inferior) | (a > superior), inferior, superior

def z_modificado(valores):
    a = np.asarray(valores, dtype=float)
    mediana = np.median(a)
    mad_bruto = stats.median_abs_deviation(a, scale=1)
    if mad_bruto == 0:
        return np.full(a.shape, np.nan)
    return 0.6745 * (a - mediana) / mad_bruto

for nome, valores in [("original", x), ("cópia corrompida", x_corrompido)]:
    tukey, inferior, superior = flags_tukey(valores)
    mz = z_modificado(valores)
    print(
        f"{nome}: cercas [{inferior:.0f}, {superior:.0f}] g; "
        f"Tukey={tukey.sum()} flag(s); |Mz|>3,5={(np.abs(mz) > 3.5).sum()} flag(s)"
    )

In [ ]:
flags_orig, li_orig, ls_orig = flags_tukey(x)
flags_corr, li_corr, ls_corr = flags_tukey(x_corrompido)
mz_orig = z_modificado(x)
mz_corr = z_modificado(x_corrompido)

assert flags_orig.sum() == 0
assert (np.abs(mz_orig) > 3.5).sum() == 0
assert flags_corr.sum() == 1
assert (np.abs(mz_corr) > 3.5).sum() == 1
assert np.isclose(mz_corr.max(), 66.269625)
print("Regras de sinalização verificadas.")

## 7. Efeito no escalonamento

Um escalonamento clássico usa média e desvio-padrão; o robusto usa mediana e IQR. Comparamos a transformação do valor típico 4.050 g e do extremo sintético. Em ML, parâmetros devem ser ajustados apenas no treino.

In [ ]:
def escala_classica(a):
    a = np.asarray(a, dtype=float)
    return (a - a.mean()) / a.std(ddof=0)

def escala_robusta(a):
    a = np.asarray(a, dtype=float)
    q1, med, q3 = np.quantile(a, [0.25, 0.50, 0.75], method="linear")
    return (a - med) / (q3 - q1)

escalas = pd.DataFrame({
    "valor_g": x_corrompido,
    "z_clássico": escala_classica(x_corrompido),
    "escala_mediana_IQR": escala_robusta(x_corrompido),
})
escalas.iloc[[np.argmin(np.abs(x_corrompido - 4050)), np.argmax(x_corrompido)]].round(4)

## 8. Curva de sensibilidade

Substituímos o mesmo ponto por valores cada vez maiores. A mediana permanece estável, enquanto a média cresce linearmente. Isso visualiza a influência de uma única observação.

In [ ]:
valores_extremos = np.array([6_300, 10_000, 20_000, 40_000, 63_000, 100_000], dtype=float)
medias, medianas = [], []
for extremo in valores_extremos:
    teste = x.copy()
    teste[np.argmax(teste)] = extremo
    medias.append(teste.mean())
    medianas.append(np.median(teste))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(valores_extremos, medias, "o-", color=azul, label="Média")
ax.plot(valores_extremos, medianas, "s--", color=laranja, label="Mediana")
ax.set(
    title="Sensibilidade a uma observação extrema",
    xlabel="Valor sintético inserido (g)",
    ylabel="Estimativa de centro (g)",
)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 9. Registro mínimo de uma decisão

Uma tabela de investigação deve separar flag, evidência e ação. Aqui, a ação é “não alterar o bruto”; em um caso real, seria necessário consultar a fonte.

In [ ]:
registro = pd.DataFrame([{
    "linha": int(indice_alvo),
    "valor_bruto_g": float(valor_original),
    "valor_teste_g": 63_000.0,
    "origem_flag": "Tukey e |z modificado| > 3,5",
    "evidência": "corrupção sintética criada neste notebook",
    "ação": "preservar bruto; usar apenas a cópia para demonstração",
    "responsável": "laboratório reproduzível",
}])
registro

## 10. Desafio

1. Escolha outra variável numérica e repita resumos e gráficos por espécie.
2. Compare o número de flags globais com o número de flags calculadas dentro de cada espécie.
3. Explique cada diferença antes de propor qualquer tratamento.
4. Modifique a função para devolver também os índices das observações sinalizadas.

Critério de qualidade: sua conclusão deve declarar (n), ausências, unidade, regra, limiar e contexto do grupo.

In [ ]:
# Verificação final do laboratório
assert df_bruto.shape == (344, 8)
assert df_bruto["body_mass_g"].notna().sum() == 342
assert df_bruto["body_mass_g"].max() == 6300
assert df_sintetico["body_mass_g"].max() == 63000
assert np.isclose(largura_fd, 343.19098643987223)
assert n_classes == 11

print("Todas as verificações passaram.")
print("Dados brutos preservados:", df_bruto["body_mass_g"].max() == 6300)

## Conclusões

- A amostra tem 344 linhas e 342 massas válidas.
- Média e desvio-padrão reagiram fortemente à corrupção; mediana, IQR, MAD e média aparada permaneceram estáveis.
- Nenhuma massa original foi sinalizada globalmente pelas duas heurísticas usadas.
- A distribuição global combina espécies diferentes; contexto de grupo é indispensável.
- Uma flag inicia uma investigação. Ela não prova erro e não autoriza exclusão.

**Fontes:** Gorman, Williams e Fraser (2014), [DOI 10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081); [palmerpenguins](https://allisonhorst.github.io/palmerpenguins/); [NIST — Detection of Outliers](https://www.itl.nist.gov/div898/handbook/eda/section3/eda35h.htm); [SciPy — median_abs_deviation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.median_abs_deviation.html).